In [1]:
# =============================================================================
# MoCo-v3 (ViT-S/16) — FULL PIPELINE: PRE-TRAINING + EVALUATION
# Pre-train on PatternNet → Evaluate on EuroSAT-RGB / EuroSAT-MS
#
# ── Comparability changes (matched to DINO / SatMAE baseline) ────────────────
#   C1.  arch: ViT-B/16 → ViT-S/16           (matches DINO / SatMAE)
#   C2.  img_size: 224  → 160                 (matches DINO S20 / SatMAE C2)
#   C3.  embed_dim: 768 → 384                 (matches DINO / SatMAE ViT-S)
#   C4.  batch_size: 4096→ 512                (matches DINO / SatMAE C5)
#   C5.  lr: 1.5e-4 → 1.5e-4 × (512/4096)    (linear-scale, ~1.9e-5 base;
#            we use 3e-4 with warmup to keep stability at smaller bs)
#   C6.  ensure_split() for consistent 80/20 eval split
#   C7.  eval crop: 224 → 160
#   C8.  timm.create_model passes img_size=160
#   C9.  student_temp: 0.2 (default) → 0.1 (original MoCo-v3 value, stable
#            with ViT-S & small bs)
#
# ── Bug fixes (parity with DINO / SatMAE fix-list) ───────────────────────────
#   FIX-1.  No channels_last anywhere — ViTs need contiguous layout.
#   FIX-2.  DDP set_device + init_process_group before model construction.
#   FIX-3.  clip_grad_norm_ inside scaler.unscale_() block.
#   FIX-4.  torch.isnan(loss) guard with informative skip.
#   FIX-5.  async_save() threading for non-blocking checkpoint I/O.
#   FIX-6.  MoCo queue initialised with F.normalize so first-batch cosine
#            sim is well-defined (was all-zeros → NaN on first InfoNCE step).
#
# ── Speed improvements (matched to DINO / SatMAE set) ────────────────────────
#   S1.  torch.compile on student + momentum encoder (reduce-overhead).
#   S2.  BF16 on Ampere+ / FP16 fallback (AMP_DTYPE).
#   S3.  TF32 matmuls (torch.set_float32_matmul_precision("high")).
#   S4.  SDPA / FlashAttention via timm set_attn_backend.
#   S5.  Fused AdamW (single CUDA kernel).
#   S6.  DDP (multi-GPU) or single-GPU training.
#   S7.  Albumentations augmentation pipeline, torchvision fallback.
#   S8.  Gradient checkpointing on student backbone.
#   S9.  foreach EMA update (_foreach_mul_ / _foreach_add_).
#   S10. Two-view DataLoader with persistent_workers + prefetch.
# =============================================================================

# !pip install timm torchvision torch scipy scikit-learn albumentations --quiet

import os, math, random, json, time, threading, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from torchvision.datasets.folder import default_loader
import timm
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# ── Albumentations (optional) ─────────────────────────────────────────────────
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBU = True
except ImportError:
    HAS_ALBU = False
    print("albumentations not found — falling back to torchvision transforms.")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True
torch.set_float32_matmul_precision("high")   # S3: TF32 on Ampere+

# ── Environment ───────────────────────────────────────────────────────────────
BASE_DIR    = "/kaggle/working"
DATA_DIR    = "/kaggle/input"
NUM_WORKERS = 4
COMPILE     = torch.__version__ >= "2.0.0"

# ── AMP dtype ─────────────────────────────────────────────────────────────────
def _amp_dtype():
    if not torch.cuda.is_available():
        return None
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 \
           else torch.float16

AMP_DTYPE = _amp_dtype()

# ── Config ────────────────────────────────────────────────────────────────────
CFG = dict(
    patternnet_dir  = f"{DATA_DIR}/datasets/samitsaleem/patternnet-scene-classification-dataset/PatternNet_Images",
    eurosat_rgb_dir = f"{DATA_DIR}/datasets/pranjallk1995/rgbeurosat/RBG",
    eurosat_ms_dir  = f"{DATA_DIR}/datasets/nguyenquangnhat2100/eurosatallbands/ds/images/remote_sensing/otherDatasets/sentinel_2/tif",
    output_dir      = f"{BASE_DIR}/mocov3",
    checkpoint      = None,

    # Architecture — scaled to ViT-S/16 @ 160 px (C1-C3, C8)
    arch          = "vit_small_patch16_224",
    img_size      = 160,           # C2: matches DINO/SatMAE
    embed_dim     = 384,           # C3: ViT-S
    proj_hidden   = 2048,          # MoCo-v3 projector hidden dim
    proj_out      = 256,           # MoCo-v3 projector output dim
    pred_hidden   = 4096,          # predictor hidden dim
    pred_out      = 256,           # predictor output dim
    momentum      = 0.996,         # EMA momentum (same schedule as DINO)
    queue_size    = 65536,

    # Training — matched to DINO/SatMAE (C4-C5)
    epochs             = 200,
    batch_size         = 512,      # C4: matches DINO/SatMAE
    lr                 = 3e-4,     # C5: stable with bs=512 + warmup
    min_lr             = 1e-6,
    weight_decay_start = 0.04,
    weight_decay_end   = 0.4,
    warmup_epochs      = 10,
    temperature        = 0.1,      # InfoNCE temperature (MoCo-v3 default)

    # Augmentation — identical to DINO/SatMAE pipeline
    jitter_strength = 0.4,
    blur_prob       = 0.5,
    global_scale    = (0.2, 1.0),

    # Evaluation — identical to DINO/SatMAE
    num_classes       = 10,
    knn_k             = 20,
    retrieval_ks      = [1, 5, 10],
    geo_thresholds_km = [1, 5, 10],
    val_split         = 0.2,

    grad_checkpoint = True,
)

os.makedirs(CFG["output_dir"], exist_ok=True)


# =============================================================================
# ── Shared utilities (identical interface to DINO/SatMAE) ────────────────────
# =============================================================================

def make_loader(ds, batch_size, shuffle, drop_last=False,
                collate_fn=None, sampler=None):
    pw = NUM_WORKERS > 0
    kwargs = dict(
        batch_size         = batch_size,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        drop_last          = drop_last,
        persistent_workers = pw,
        prefetch_factor    = 4 if pw else None,
    )
    if sampler is not None:
        kwargs["sampler"] = sampler
    else:
        kwargs["shuffle"] = shuffle
    if collate_fn is not None:
        kwargs["collate_fn"] = collate_fn
    return DataLoader(ds, **kwargs)


def ensure_split(root, val_frac=0.2, seed=SEED):
    """Identical to DINO/SatMAE ensure_split — guarantees same 80/20 split."""
    if os.path.isdir(os.path.join(root, "train")):
        return root
    split_root = root.rstrip("/") + "_split"
    if os.path.isdir(os.path.join(split_root, "train")):
        print(f"  Using cached split at {split_root}")
        return split_root
    print(f"  Creating 80/20 stratified split → {split_root} …")
    base   = datasets.ImageFolder(root)
    labels = np.array([y for _, y in base.samples])
    sss    = StratifiedShuffleSplit(n_splits=1, test_size=val_frac,
                                    random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))
    for split_name, indices in [("train", train_idx), ("val", val_idx)]:
        for idx in indices:
            src_path, cls_idx = base.samples[idx]
            cls_name = base.classes[cls_idx]
            dst_dir  = os.path.join(split_root, split_name, cls_name)
            os.makedirs(dst_dir, exist_ok=True)
            dst_path = os.path.join(dst_dir, os.path.basename(src_path))
            if not os.path.exists(dst_path):
                try:
                    os.link(src_path, dst_path)
                except OSError:
                    shutil.copy2(src_path, dst_path)
    print(f"  Split: {len(train_idx)} train / {len(val_idx)} val")
    return split_root


_save_thread: threading.Thread = None

def async_save(path, obj):
    """FIX-5: non-blocking checkpoint I/O."""
    global _save_thread
    if _save_thread is not None:
        _save_thread.join()
    def _save():
        torch.save(obj, path)
        print(f"  → Saved {path}", flush=True)
    _save_thread = threading.Thread(target=_save, daemon=True)
    _save_thread.start()


def cosine_schedule(start, end, epoch, total):
    return end + 0.5 * (start - end) * (1.0 + math.cos(math.pi * epoch / total))


# =============================================================================
# ── Augmentation pipeline (identical to DINO/SatMAE) ─────────────────────────
# =============================================================================

def _base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        A.RandomResizedCrop(size=(size, size), scale=scale, interpolation=3),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=jitter_s, contrast=jitter_s,
                      saturation=jitter_s, hue=jitter_s * 0.25, p=0.8),
        A.ToGray(p=0.2),
        A.GaussianBlur(blur_limit=(9, 9), sigma_limit=(0.1, 2.0), p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(A.Solarize(threshold=128, p=solarize_prob))
    ops += [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    return A.Compose(ops)


def _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    ops = [
        transforms.RandomResizedCrop(size, scale=scale, interpolation=3),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(
            brightness=jitter_s, contrast=jitter_s,
            saturation=jitter_s, hue=jitter_s * 0.25)], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0))],
            p=blur_prob),
    ]
    if solarize_prob > 0:
        ops.append(transforms.RandomSolarize(threshold=128, p=solarize_prob))
    ops += [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
    return transforms.Compose(ops)


def _base_aug(size, scale, jitter_s, blur_prob, solarize_prob=0.0):
    return (_base_aug_albu(size, scale, jitter_s, blur_prob, solarize_prob)
            if HAS_ALBU else
            _base_aug_tv(size, scale, jitter_s, blur_prob, solarize_prob))


# =============================================================================
# ── Datasets ──────────────────────────────────────────────────────────────────
# =============================================================================

class TwoViewDataset(Dataset):
    """Two independently augmented views of each image for contrastive learning."""
    def __init__(self, root, aug):
        self.base     = datasets.ImageFolder(root)
        self.aug      = aug
        self.use_albu = HAS_ALBU

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        if self.use_albu:
            arr = np.array(img)
            x1  = self.aug(image=arr)["image"]
            x2  = self.aug(image=arr)["image"]
        else:
            x1 = self.aug(img)
            x2 = self.aug(img)
        return x1, x2, label


class MultiSpectralDataset(Dataset):
    """Supports .tif (rasterio), .npy, .png/.jpg — identical to SatMAE."""
    def __init__(self, root, n_channels=13):
        self.samples    = []
        self.n_channels = n_channels
        classes = sorted(d for d in os.listdir(root)
                         if os.path.isdir(os.path.join(root, d)))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(root, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(
                        (".tif", ".tiff", ".npy", ".png", ".jpg")):
                    self.samples.append(
                        (os.path.join(cls_dir, fname),
                         self.class_to_idx[cls]))

    def __len__(self): return len(self.samples)

    def _load_tif(self, path):
        try:
            import rasterio
        except ImportError:
            raise ImportError("pip install rasterio")
        with rasterio.open(path) as src:
            arr = src.read().astype(np.float32)
        n = self.n_channels
        if arr.shape[0] >= n:
            arr = arr[:n]
        else:
            pad = np.zeros((n - arr.shape[0], *arr.shape[1:]), dtype=np.float32)
            arr = np.concatenate([arr, pad], axis=0)
        return torch.from_numpy(arr)

    def _normalise(self, x):
        mean = x.view(x.shape[0], -1).mean(1, keepdim=True).unsqueeze(-1)
        std  = (x.view(x.shape[0], -1).std(1, keepdim=True)
                 .unsqueeze(-1).clamp(min=1e-6))
        return (x - mean) / std

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        ext = os.path.splitext(path)[1].lower()
        if ext in (".tif", ".tiff"):
            x = self._load_tif(path)
        elif ext == ".npy":
            arr = np.load(path).astype(np.float32)
            if arr.ndim == 3 and arr.shape[2] == self.n_channels:
                arr = arr.transpose(2, 0, 1)
            x = torch.from_numpy(arr)
            if x.shape[0] > self.n_channels:
                x = x[:self.n_channels]
        else:
            from PIL import Image
            img = Image.open(path).convert("RGB")
            img_size = CFG["img_size"]
            resize   = int(img_size * (256 / 224) + 0.5)
            x = transforms.Compose([
                transforms.Resize(resize),
                transforms.CenterCrop(img_size),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406],
                                      [0.229, 0.224, 0.225]),
            ])(img)
            x = x.repeat(math.ceil(self.n_channels / 3), 1, 1)[:self.n_channels]
            return x, label
        x = self._normalise(x)
        x = F.interpolate(x[None], size=CFG["img_size"],
                          mode="bilinear", align_corners=False)[0]
        return x, label


# =============================================================================
# ── Backbone factory ─────────────────────────────────────────────────────────
# =============================================================================

def _make_backbone(arch=None, img_size=None):
    arch     = arch     or CFG["arch"]
    img_size = img_size or CFG["img_size"]
    m = timm.create_model(
        arch,
        pretrained=False,
        num_classes=0,
        img_size=img_size,
        dynamic_img_size=True,
    )
    if hasattr(m, "set_attn_backend"):  # S4: SDPA/FlashAttention
        try:
            m.set_attn_backend("sdpa")
        except Exception:
            pass
    return m


# =============================================================================
# ── MoCo-v3 Projector / Predictor heads ──────────────────────────────────────
# =============================================================================

class MoCoProjector(nn.Module):
    """Three-layer MLP with BN (same design as MoCo-v3 paper)."""
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim, bias=False),
            nn.BatchNorm1d(out_dim, affine=False),
        )

    def forward(self, x):
        return self.net(x)


class MoCoPredictor(nn.Module):
    """Two-layer MLP predictor (used only in student path)."""
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


# =============================================================================
# ── MoCo-v3 Model ─────────────────────────────────────────────────────────────
# =============================================================================

class MoCoV3(nn.Module):
    """
    MoCo-v3 with ViT-S/16 backbone.

    Architecture:
      student:  backbone_q → projector_q → predictor  →  query q
      momentum: backbone_k → projector_k              →  key k

    Loss: symmetric InfoNCE (query vs. queue of keys).

    FIX-6: queue is initialised with F.normalize'd random vectors so
    cosine similarities on the very first batch are well-defined.
    """

    def __init__(self, cfg):
        super().__init__()
        self.temperature = cfg["temperature"]

        # Student (query) path
        self.backbone_q  = _make_backbone(cfg["arch"], cfg["img_size"])
        self.projector_q = MoCoProjector(cfg["embed_dim"],
                                          cfg["proj_hidden"], cfg["proj_out"])
        self.predictor   = MoCoPredictor(cfg["proj_out"],
                                          cfg["pred_hidden"], cfg["pred_out"])

        # Momentum (key) path — no gradients
        self.backbone_k  = _make_backbone(cfg["arch"], cfg["img_size"])
        self.projector_k = MoCoProjector(cfg["embed_dim"],
                                          cfg["proj_hidden"], cfg["proj_out"])
        for p in (list(self.backbone_k.parameters()) +
                  list(self.projector_k.parameters())):
            p.requires_grad_(False)

        # Cosine-similarity queue  (FIX-6: normalised init)
        self.register_buffer("queue",
            F.normalize(torch.randn(cfg["proj_out"], cfg["queue_size"]), dim=0))
        self.register_buffer("queue_ptr", torch.zeros(1, dtype=torch.long))
        self.K = cfg["queue_size"]

        # Copy weights to momentum encoder
        self._copy_weights(self.backbone_q,  self.backbone_k)
        self._copy_weights(self.projector_q, self.projector_k)

        # S8: gradient checkpointing on student backbone
        if cfg.get("grad_checkpoint") and hasattr(
                self.backbone_q, "set_grad_checkpointing"):
            self.backbone_q.set_grad_checkpointing(True)

        # S9: cache param lists for foreach EMA
        self._q_bb_params = list(self.backbone_q.parameters())
        self._k_bb_params = list(self.backbone_k.parameters())
        self._q_pj_params = list(self.projector_q.parameters())
        self._k_pj_params = list(self.projector_k.parameters())

    @staticmethod
    def _copy_weights(src, dst):
        for sp, dp in zip(src.parameters(), dst.parameters()):
            dp.data.copy_(sp.data)

    # S9: foreach EMA
    @torch.no_grad()
    def _update_momentum(self, m):
        alpha = 1.0 - m
        torch._foreach_mul_(self._k_bb_params, m)
        torch._foreach_add_(self._k_bb_params, self._q_bb_params, alpha=alpha)
        torch._foreach_mul_(self._k_pj_params, m)
        torch._foreach_add_(self._k_pj_params, self._q_pj_params, alpha=alpha)

    @torch.no_grad()
    def _dequeue_and_enqueue(self, keys):
        """Replace oldest queue entries with the current batch of keys."""
        batch_size = keys.shape[0]
        ptr = int(self.queue_ptr)
        # Wrap around if needed
        end = ptr + batch_size
        if end <= self.K:
            self.queue[:, ptr:end] = keys.T
        else:
            # Split across boundary
            overflow = end - self.K
            self.queue[:, ptr:]  = keys[:batch_size - overflow].T
            self.queue[:, :overflow] = keys[batch_size - overflow:].T
        self.queue_ptr[0] = end % self.K

    @staticmethod
    def _info_nce(q, k, queue, temperature):
        """
        Symmetric InfoNCE loss.
        q, k: (B, D) normalised
        queue: (D, K)
        """
        B = q.shape[0]
        # Positive logit: dot(q, k)
        l_pos = (q * k).sum(dim=-1, keepdim=True)           # (B, 1)
        # Negative logits: dot(q, queue)
        l_neg = q @ queue.clone().detach()                   # (B, K)
        logits = torch.cat([l_pos, l_neg], dim=1) / temperature  # (B, 1+K)
        labels = torch.zeros(B, dtype=torch.long, device=q.device)
        return F.cross_entropy(logits, labels)

    def forward(self, x1, x2, momentum):
        """
        x1, x2: two views of the same images, shape (B, 3, H, W).
        momentum: current EMA coefficient.
        FIX-1: inputs must be plain contiguous tensors (no channels_last).
        """
        # ── Student (query) path ──────────────────────────────────────────────
        q1 = self.predictor(F.normalize(
            self.projector_q(self.backbone_q(x1)), dim=-1))
        q2 = self.predictor(F.normalize(
            self.projector_q(self.backbone_q(x2)), dim=-1))
        q1 = F.normalize(q1, dim=-1)
        q2 = F.normalize(q2, dim=-1)

        # ── Momentum (key) path ──────────────────────────────────────────────
        with torch.no_grad():
            self._update_momentum(momentum)
            k1 = F.normalize(self.projector_k(self.backbone_k(x1)), dim=-1)
            k2 = F.normalize(self.projector_k(self.backbone_k(x2)), dim=-1)

        # Symmetric InfoNCE: q1 vs k2-queue, q2 vs k1-queue
        loss = 0.5 * (self._info_nce(q1, k2, self.queue, self.temperature) +
                      self._info_nce(q2, k1, self.queue, self.temperature))

        # Update queue with both key batches
        with torch.no_grad():
            self._dequeue_and_enqueue(k1)
            self._dequeue_and_enqueue(k2)

        return loss


# =============================================================================
# ── Schedule helpers (identical to DINO) ─────────────────────────────────────
# =============================================================================

def build_epoch_schedules(cfg):
    epochs          = cfg["epochs"]
    lrs, wds, momenta = [], [], []
    for e in range(epochs):
        if e < cfg["warmup_epochs"]:
            lr = cfg["lr"] * (e + 1) / cfg["warmup_epochs"]
        else:
            lr = cosine_schedule(cfg["lr"], cfg["min_lr"],
                                 e - cfg["warmup_epochs"],
                                 epochs - cfg["warmup_epochs"])
        wd  = cosine_schedule(cfg["weight_decay_start"],
                               cfg["weight_decay_end"], e, epochs)
        mom = cosine_schedule(cfg["momentum"], 1.0, e, epochs)
        lrs.append(lr); wds.append(wd); momenta.append(mom)
    return lrs, wds, momenta


def apply_lr_wd(optimizer, lr, wd):
    for g in optimizer.param_groups:
        g["lr"] = lr
        if g.get("apply_wd", True):
            g["weight_decay"] = wd


# =============================================================================
# ── Training loop (DDP-aware) ─────────────────────────────────────────────────
# =============================================================================

def train(rank: int, world_size: int):
    # FIX-2: set device and init process group FIRST
    torch.cuda.set_device(rank)
    device  = torch.device(f"cuda:{rank}")
    is_ddp  = world_size > 1
    is_main = rank == 0

    if is_ddp:
        dist.init_process_group(
            backend="nccl", init_method="env://",
            world_size=world_size, rank=rank)

    if is_main:
        print(f"Device: {device}  |  World: {world_size}  |  "
              f"Workers: {NUM_WORKERS}  |  compile: {COMPILE}  |  "
              f"AMP: {AMP_DTYPE}  |  albu: {HAS_ALBU}", flush=True)

    # ── Data ──────────────────────────────────────────────────────────────────
    aug = _base_aug(CFG["img_size"], CFG["global_scale"],
                    CFG["jitter_strength"], CFG["blur_prob"])
    dataset = TwoViewDataset(CFG["patternnet_dir"], aug)
    sampler = (DistributedSampler(dataset, num_replicas=world_size,
                                   rank=rank, shuffle=True, drop_last=True)
               if is_ddp else None)
    loader  = make_loader(dataset, CFG["batch_size"],
                          shuffle=not is_ddp, drop_last=True,
                          sampler=sampler)

    # ── Model ─────────────────────────────────────────────────────────────────
    # FIX-1: no channels_last for ViT
    model = MoCoV3(CFG).to(device)

    # S1: compile BEFORE DDP so both compile and DDP see the same module graph
    if COMPILE:
        try:
            model.backbone_q = torch.compile(
                model.backbone_q, mode="reduce-overhead")
            model.backbone_k = torch.compile(
                model.backbone_k, mode="reduce-overhead")
            if is_main:
                print("torch.compile enabled on student + momentum backbone.")
        except Exception as e:
            if is_main:
                print(f"torch.compile skipped: {e}")

    if is_ddp:
        # Only student parts need DDP (momentum encoder has no gradients)
        model.backbone_q  = DDP(model.backbone_q,  device_ids=[rank],
                                 find_unused_parameters=False)
        model.projector_q = DDP(model.projector_q, device_ids=[rank],
                                 find_unused_parameters=False)
        model.predictor   = DDP(model.predictor,   device_ids=[rank],
                                 find_unused_parameters=False)

    # ── Optimizer (S5: fused AdamW) ───────────────────────────────────────────
    wd_params = [p for n, p in model.named_parameters()
                 if p.requires_grad and "bias" not in n and "norm" not in n]
    no_wd     = [p for n, p in model.named_parameters()
                 if p.requires_grad and ("bias" in n or "norm" in n)]
    optimizer = torch.optim.AdamW(
        [{"params": wd_params, "apply_wd": True},
         {"params": no_wd,     "apply_wd": False, "weight_decay": 0.0}],
        lr=CFG["lr"], weight_decay=CFG["weight_decay_start"],
        fused=True)

    use_amp    = AMP_DTYPE is not None
    use_scaler = AMP_DTYPE == torch.float16
    scaler     = torch.amp.GradScaler("cuda", enabled=use_scaler)
    trainable  = [p for p in model.parameters() if p.requires_grad]

    lrs, wds, momenta = build_epoch_schedules(CFG)
    start_epoch = 0

    if CFG["checkpoint"] and is_main:
        ckpt = torch.load(CFG["checkpoint"], map_location="cpu")
        model.load_state_dict(ckpt["model"], strict=False)
        optimizer.load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from epoch {start_epoch}")

    log           = []
    step_counter  = 0
    t_train_start = time.time()

    for epoch in range(start_epoch, CFG["epochs"]):
        if is_ddp:
            sampler.set_epoch(epoch)
        apply_lr_wd(optimizer, lrs[epoch], wds[epoch])
        momentum = momenta[epoch]

        model.train()
        total_loss = 0.0
        t0         = time.time()

        for x1, x2, _ in loader:
            # FIX-1: plain .to(device) — no channels_last
            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                loss = model(x1, x2, momentum)

            # FIX-4: guard against NaN/Inf loss
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"[rank {rank}] NaN/Inf at step {step_counter}, "
                      f"epoch {epoch+1} — skipping batch.", flush=True)
                optimizer.zero_grad(set_to_none=True)
                step_counter += 1
                continue

            scaler.scale(loss).backward()
            # FIX-3: unscale before clip so norm is on true gradient scale
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(trainable, max_norm=3.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()
            step_counter += 1

        avg_loss = total_loss / max(len(loader), 1)
        elapsed  = time.time() - t0

        if is_main:
            log.append({"epoch": epoch, "loss": avg_loss,
                        "lr": lrs[epoch], "momentum": momentum,
                        "epoch_time_s": round(elapsed, 1)})
            print(f"Epoch [{epoch+1:>3}/{CFG['epochs']}]  "
                  f"loss={avg_loss:.4f}  lr={lrs[epoch]:.2e}  "
                  f"time={elapsed:.0f}s")

            if (epoch + 1) % 50 == 0 or epoch == CFG["epochs"] - 1:
                raw = model
                state = {k.replace("module.", ""): v
                         for k, v in raw.state_dict().items()}
                path = os.path.join(CFG["output_dir"],
                                     f"mocov3_ep{epoch+1}.pt")
                async_save(path, {"epoch": epoch, "model": state,
                                   "optimizer": optimizer.state_dict(),
                                   "cfg": CFG})

    if is_main:
        if _save_thread is not None:
            _save_thread.join()
        with open(os.path.join(CFG["output_dir"], "mocov3_log.json"), "w") as f:
            json.dump(log, f, indent=2)
        total = time.time() - t_train_start
        print(f"\nPre-training done.  Total: {total/3600:.2f} h ({total:.0f} s)")

    if is_ddp:
        dist.destroy_process_group()


# =============================================================================
# ── Backbone loading (eval) ───────────────────────────────────────────────────
# =============================================================================

def load_backbone(backbone_path, device):
    """Load frozen MoCo-v3 student backbone (backbone_q).
    FIX-1: plain .to() — no channels_last for ViT eval."""
    ckpt  = torch.load(backbone_path, map_location="cpu")
    state = {k.replace("module.", "").replace("backbone_q.", ""): v
             for k, v in ckpt["model"].items() if "backbone_q" in k}
    backbone = _make_backbone()
    backbone.load_state_dict(state, strict=False)
    backbone.eval().to(device)
    for p in backbone.parameters():
        p.requires_grad_(False)
    return backbone


def get_eval_transforms():
    """C7: eval crop at CFG img_size (160), not hardcoded 224."""
    img_size = CFG["img_size"]
    resize   = int(img_size * (256 / 224) + 0.5)
    val_tf = transforms.Compose([
        transforms.Resize(resize),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf


@torch.inference_mode()
def extract_features(backbone, loader, device):
    all_feats, all_labels = [], []
    backbone.eval()
    use_amp = AMP_DTYPE is not None
    for x, y in loader:
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
            feats = backbone(x.to(device, non_blocking=True))
        all_feats.append(feats.float().cpu())
        all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


# =============================================================================
# ── Shared eval helpers ───────────────────────────────────────────────────────
# =============================================================================

def batched_knn_predict(sim, train_labels, k, num_classes, chunk=512):
    N_val  = sim.shape[0]
    device = sim.device
    preds  = torch.empty(N_val, dtype=torch.long, device=device)
    for start in range(0, N_val, chunk):
        end        = min(start + chunk, N_val)
        top_idx    = sim[start:end].topk(k, dim=1).indices
        top_labels = train_labels.to(device)[top_idx]
        B          = end - start
        votes      = torch.zeros(B, num_classes, device=device)
        votes.scatter_add_(1, top_labels,
                           torch.ones(B, k, device=device))
        preds[start:end] = votes.argmax(1)
    return preds


def effective_rank(feats):
    f       = feats - feats.mean(0)
    cov     = (f.T @ f) / (feats.shape[0] - 1)
    eigvals = torch.linalg.eigvalsh(cov.float()).clamp(min=0)
    eigvals = eigvals / eigvals.sum().clamp(min=1e-8)
    eigvals = eigvals[eigvals > 1e-9]
    entropy = -(eigvals * eigvals.log()).sum()
    return math.exp(entropy.item())


def uniformity_score(feats):
    feats = F.normalize(feats.float(), dim=-1)
    if feats.shape[0] > 2000:
        feats = feats[torch.randperm(feats.shape[0])[:2000]]
    sq = torch.cdist(feats, feats, p=2).pow(2)
    return round(sq.mul(-2).exp().mean().log().item(), 4)


def haversine_km(lat1, lon1, lat2, lon2):
    R  = 6371.0
    dr = math.radians
    dlat = dr(lat2 - lat1); dlon = dr(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2 +
         math.cos(dr(lat1)) * math.cos(dr(lat2)) *
         math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))


def mean_average_precision(sim_matrix, labels):
    N  = sim_matrix.shape[0]
    sm = sim_matrix.clone()
    sm.fill_diagonal_(-1e9)
    order   = sm.argsort(dim=1, descending=True)
    ap_list = []
    for i in range(N):
        gt    = (labels[order[i]] == labels[i])
        n_rel = gt.sum().item()
        if n_rel == 0: continue
        n_correct = 0; precisions = []
        for rank_, hit in enumerate(gt.tolist(), 1):
            if hit:
                n_correct += 1
                precisions.append(n_correct / rank_)
        ap_list.append(sum(precisions) / n_rel)
    return float(np.mean(ap_list)) * 100 if ap_list else 0.0


# =============================================================================
# §4.1  CLASSIFICATION — linear probe at multiple label fractions
# =============================================================================

def eval_classification(backbone_path, eurosat_dir,
                         label_fracs=(0.01, 0.1, 1.0), num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    train_tf, val_tf = get_eval_transforms()
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])

    train_full = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=train_tf)
    val_ds     = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)

    print("  Pre-extracting features for linear probe…")
    all_train_feats, all_train_labels = extract_features(
        backbone, make_loader(train_full, 512, shuffle=False), device)
    val_feats, val_labels = extract_features(
        backbone, make_loader(val_ds, 512, shuffle=False), device)

    all_train_feats  = all_train_feats.to(device)
    all_train_labels = all_train_labels.to(device)
    val_feats_dev    = val_feats.to(device)
    val_labels_dev   = val_labels.to(device)

    results = {}
    bs      = 256

    for frac in label_fracs:
        n       = max(num_classes, int(len(train_full) * frac))
        indices = random.sample(range(len(all_train_feats)), n)
        idx_t   = torch.tensor(indices, device=device)
        f_sub   = all_train_feats[idx_t]
        l_sub   = all_train_labels[idx_t]

        head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
        opt   = torch.optim.SGD(head.parameters(), lr=0.1,
                                 momentum=0.9, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100)

        for _ in range(100):
            head.train()
            perm = torch.randperm(len(f_sub), device=device)
            for start in range(0, len(f_sub), bs):
                sel = perm[start:start + bs]
                opt.zero_grad()
                F.cross_entropy(head(f_sub[sel]), l_sub[sel]).backward()
                opt.step()
            sched.step()

        head.eval()
        with torch.no_grad():
            preds = torch.cat([head(val_feats_dev[s:s + bs]).argmax(1)
                               for s in range(0, len(val_feats_dev), bs)]
                              ).cpu().numpy()
        labels = val_labels_dev.cpu().numpy()

        acc      = 100.0 * (preds == labels).mean()
        macro_f1 = 100.0 * f1_score(labels, preds, average="macro")
        key = f"{int(round(frac * 100))}pct"
        results[key] = {"top1_acc": round(acc, 2),
                        "macro_f1": round(macro_f1, 2)}
        print(f"  [{int(frac*100)}% labels]  "
              f"Top-1={acc:.2f}%  Macro-F1={macro_f1:.2f}%")

    return results


# =============================================================================
# §4.2  SEGMENTATION — frozen backbone + linear pixel decoder
# =============================================================================

class SegDecoder(nn.Module):
    def __init__(self, embed_dim, num_classes, patch_size=16, img_size=160):
        super().__init__()
        self.grid_size = img_size // patch_size
        self.img_size  = img_size
        self.head      = nn.Conv2d(embed_dim, num_classes, kernel_size=1)

    def forward(self, patch_tokens):
        B, N, D = patch_tokens.shape
        g = self.grid_size
        x = patch_tokens.permute(0, 2, 1).reshape(B, D, g, g)
        x = self.head(x)
        return F.interpolate(x, size=(self.img_size, self.img_size),
                             mode="bilinear", align_corners=False)


def boundary_f1(pred_mask, gt_mask, num_classes, dilation=1):
    from scipy.ndimage import binary_dilation as bd
    bf1_list = []
    for c in range(num_classes):
        p = (pred_mask == c).astype(np.uint8)
        g = (gt_mask   == c).astype(np.uint8)
        if g.sum() == 0: continue
        p_b   = np.logical_xor(p, bd(p, iterations=dilation)).astype(np.uint8)
        g_b   = np.logical_xor(g, bd(g, iterations=dilation)).astype(np.uint8)
        inter = (p_b & g_b).sum(); denom = p_b.sum() + g_b.sum()
        if denom == 0: continue
        bf1_list.append(2 * inter / (denom + 1e-8))
    return float(np.mean(bf1_list)) if bf1_list else 0.0


def eval_segmentation(backbone_path, eurosat_dir, num_classes=10, epochs=30):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()
    aug_tf    = transforms.Compose([
        transforms.RandomResizedCrop(CFG["img_size"]),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=aug_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 64, shuffle=True,  drop_last=True)
    val_ldr   = make_loader(val_ds,   64, shuffle=False)

    decoder = SegDecoder(CFG["embed_dim"], num_classes,
                          patch_size=16, img_size=CFG["img_size"]).to(device)
    opt     = torch.optim.Adam(decoder.parameters(), lr=1e-3)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    use_amp = AMP_DTYPE is not None

    def get_patch_tokens(x):
        with torch.inference_mode():
            # FIX-1: no channels_last
            out = backbone.forward_features(
                x.to(device, non_blocking=True))
            if isinstance(out, torch.Tensor) and out.dim() == 3:
                return out[:, 1:]   # drop CLS token
            return backbone(x).unsqueeze(1).expand(
                -1, (CFG["img_size"] // 16) ** 2, -1)

    for epoch in range(epochs):
        decoder.train()
        for x, y in train_ldr:
            tokens = get_patch_tokens(x)
            y      = y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=use_amp):
                logits = decoder(tokens)
                target = y.view(-1, 1, 1).expand(
                    -1, CFG["img_size"], CFG["img_size"])
                loss   = F.cross_entropy(logits, target)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Seg epoch {epoch+1}/{epochs}", flush=True)

    decoder.eval()
    confusion = np.zeros((num_classes, num_classes), dtype=np.int64)
    all_bf1   = []
    with torch.inference_mode():
        for x, y in val_ldr:
            tokens = get_patch_tokens(x)
            pred   = decoder(tokens).argmax(1).cpu().numpy()
            label  = y.numpy()
            for b in range(pred.shape[0]):
                gt_map = np.full_like(pred[b], label[b])
                for i in range(num_classes):
                    for j in range(num_classes):
                        confusion[i, j] += (
                            (gt_map == i) & (pred[b] == j)).sum()
                all_bf1.append(boundary_f1(pred[b], gt_map, num_classes))

    iou_per_class = []
    for c in range(num_classes):
        tp    = confusion[c, c]
        fp    = confusion[:, c].sum() - tp
        fn    = confusion[c, :].sum() - tp
        denom = tp + fp + fn
        if denom > 0:
            iou_per_class.append(tp / denom)

    miou     = float(np.mean(iou_per_class)) * 100
    mean_bf1 = float(np.mean(all_bf1)) * 100
    print(f"  Segmentation  mIoU={miou:.2f}%  Boundary-F1={mean_bf1:.2f}%")
    return {"miou": round(miou, 2), "boundary_f1": round(mean_bf1, 2)}


# =============================================================================
# §4.3  RETRIEVAL PERFORMANCE
# =============================================================================

def eval_retrieval(backbone_path, eurosat_dir, gps_csv=None,
                    ks=(1, 5, 10), geo_thresholds_km=(1, 5, 10)):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)

    print("  Extracting gallery features…")
    feats, labels = extract_features(backbone, val_ldr, device)
    feats = F.normalize(feats.float(), dim=-1)
    sim   = feats @ feats.T
    N     = feats.shape[0]

    results     = {}
    sim_no_diag = sim.clone()
    sim_no_diag.fill_diagonal_(-1e9)
    topk_max    = max(ks)
    top_indices = sim_no_diag.topk(topk_max, dim=1).indices

    for k in ks:
        top_k_labels = labels[top_indices[:, :k]]
        correct = (top_k_labels == labels.unsqueeze(1)).any(dim=1).sum().item()
        recall  = 100.0 * correct / N
        results[f"recall@{k}"] = round(recall, 2)
        print(f"  Recall@{k} = {recall:.2f}%")

    mAP = mean_average_precision(sim, labels)
    results["mAP"] = round(mAP, 2)
    print(f"  mAP = {mAP:.2f}%")

    if gps_csv is not None:
        import pandas as pd
        gps_df       = pd.read_csv(gps_csv)
        img_paths    = [val_ds.samples[i][0] for i in range(N)]
        fname_to_gps = {row["filename"]: (row["lat"], row["lon"])
                        for _, row in gps_df.iterrows()}
        errors_km = []
        geo_hits  = {k: {eps: 0 for eps in geo_thresholds_km} for k in ks}
        n_valid   = 0

        for i in range(N):
            q_fname = os.path.basename(img_paths[i])
            if q_fname not in fname_to_gps: continue
            n_valid += 1
            q_lat, q_lon = fname_to_gps[q_fname]
            top_fnames   = [os.path.basename(img_paths[j])
                            for j in top_indices[i, :topk_max].tolist()]
            if top_fnames[0] in fname_to_gps:
                r_lat, r_lon = fname_to_gps[top_fnames[0]]
                errors_km.append(haversine_km(q_lat, q_lon, r_lat, r_lon))
            for k in ks:
                cands = [f for f in top_fnames[:k] if f in fname_to_gps]
                for eps in geo_thresholds_km:
                    if any(haversine_km(q_lat, q_lon, *fname_to_gps[f]) <= eps
                           for f in cands):
                        geo_hits[k][eps] += 1

        if errors_km:
            med_err = float(np.median(errors_km)) * 1000
            p90_err = float(np.percentile(errors_km, 90)) * 1000
            results["median_loc_error_m"] = round(med_err, 1)
            results["p90_loc_error_m"]    = round(p90_err, 1)
            print(f"  Median loc. error = {med_err:.1f} m  "
                  f"(P90 = {p90_err:.1f} m)")

        if n_valid > 0:
            for k in ks:
                for eps in geo_thresholds_km:
                    r = 100.0 * geo_hits[k][eps] / n_valid
                    results[f"geo_recall@{k}_{eps}km"] = round(r, 2)
                    print(f"  Geo-Recall@{k} ({eps} km) = {r:.2f}%")

    return results


# =============================================================================
# §4.4  REPRESENTATION QUALITY
# =============================================================================

def eval_representation_quality(backbone_path, eurosat_dir,
                                  knn_k=20, num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    train_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "train"), transform=val_tf)
    val_ds    = datasets.ImageFolder(
        os.path.join(split_dir, "val"),   transform=val_tf)
    train_ldr = make_loader(train_ds, 256, shuffle=False)
    val_ldr   = make_loader(val_ds,   256, shuffle=False)

    print("  Extracting features for representation quality…")
    train_feats, train_labels = extract_features(backbone, train_ldr, device)
    val_feats,   val_labels   = extract_features(backbone, val_ldr,   device)
    train_n = F.normalize(train_feats.float(), dim=-1)
    val_n   = F.normalize(val_feats.float(),   dim=-1)

    sim       = val_n @ train_n.T
    knn_preds = batched_knn_predict(
        sim.to(device), train_labels, knn_k, num_classes)
    knn_acc   = 100.0 * (knn_preds.cpu() == val_labels).float().mean().item()
    print(f"  kNN accuracy (k={knn_k}) = {knn_acc:.2f}%")

    eff_rank = effective_rank(val_feats.float())
    print(f"  Effective rank = {eff_rank:.1f}")

    unif = uniformity_score(val_n)
    print(f"  Uniformity = {unif:.4f}")

    aug = _base_aug(CFG["img_size"], CFG["global_scale"],
                    CFG["jitter_strength"], CFG["blur_prob"])
    two_view_ds  = TwoViewDataset(os.path.join(split_dir, "val"), aug)
    two_view_ldr = make_loader(two_view_ds, 256, shuffle=False)
    align_scores = []
    with torch.inference_mode():
        for x1, x2, _ in two_view_ldr:
            # FIX-1: no channels_last
            z1 = F.normalize(backbone(x1.to(device, non_blocking=True)),
                             dim=-1)
            z2 = F.normalize(backbone(x2.to(device, non_blocking=True)),
                             dim=-1)
            align_scores.append(
                (z1 - z2).pow(2).sum(dim=-1).mean().item())
    alignment = round(float(np.mean(align_scores)), 4)
    print(f"  Alignment = {alignment:.4f}")

    return {"knn_acc":        round(knn_acc, 2),
            "effective_rank": round(eff_rank, 1),
            "uniformity":     unif,
            "alignment":      alignment}


# =============================================================================
# §4.5  BAND MISMATCH ROBUSTNESS
# =============================================================================

class BandAdapterBackbone(nn.Module):
    def __init__(self, backbone, in_channels=13, out_channels=3):
        super().__init__()
        self.adapter  = nn.Conv2d(in_channels, out_channels,
                                   kernel_size=1, bias=False)
        self.backbone = backbone
        nn.init.kaiming_normal_(self.adapter.weight)

    def forward(self, x):
        return self.backbone(self.adapter(x))


def eval_band_mismatch(backbone_path, eurosat_rgb_dir,
                        eurosat_ms_dir=None, num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(backbone, val_ldr, device)
    feats  = F.normalize(feats.float(), dim=-1)
    sim_nd = feats @ feats.T
    sim_nd.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (
        labels[sim_nd.argmax(dim=1)] == labels).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(eurosat_ms_dir, n_channels=13)
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb   = BandAdapterBackbone(
                backbone, in_channels=13, out_channels=3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)

            tmp_head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) +
                list(tmp_head.parameters()), lr=1e-3)

            for _ in range(5):
                adapter_bb.adapter.train(); tmp_head.train()
                for x, y in ms_ldr_train:
                    x, y = (x.to(device, non_blocking=True),
                            y.to(device, non_blocking=True))
                    with torch.inference_mode():
                        feat = backbone(adapter_bb.adapter(x))
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            adapter_bb.eval()
            ms_ldr_eval      = make_loader(ms_ds, 128, shuffle=False)
            ms_feats, ms_lbl = [], []
            with torch.inference_mode():
                for x, y in ms_ldr_eval:
                    ms_feats.append(
                        adapter_bb(x.to(device, non_blocking=True))
                        .float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (
                ms_labels[sim_ms.argmax(dim=1)] == ms_labels
            ).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = round(recall_rgb - recall_ms, 2) if recall_ms is not None else None
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")

    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms is not None
                               else None,
        "band_mismatch_delta": delta,
    }


# =============================================================================
# ── Full evaluation orchestrator ──────────────────────────────────────────────
# =============================================================================

def run_full_evaluation(backbone_path, eurosat_rgb_dir=None,
                         eurosat_ms_dir=None, gps_csv=None):
    eurosat_rgb_dir = eurosat_rgb_dir or CFG["eurosat_rgb_dir"]
    eurosat_ms_dir  = eurosat_ms_dir  or CFG.get("eurosat_ms_dir")
    all_results     = {"model": "MoCo-v3-ViT-S", "checkpoint": backbone_path}

    print("\n" + "=" * 60)
    print("§4.1  CLASSIFICATION (linear probe)")
    print("=" * 60)
    all_results["classification"] = eval_classification(
        backbone_path, eurosat_rgb_dir,
        label_fracs=(0.01, 0.1, 1.0), num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.2  SEGMENTATION")
    print("=" * 60)
    all_results["segmentation"] = eval_segmentation(
        backbone_path, eurosat_rgb_dir, num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.3  RETRIEVAL PERFORMANCE")
    print("=" * 60)
    all_results["retrieval"] = eval_retrieval(
        backbone_path, eurosat_rgb_dir, gps_csv=gps_csv,
        ks=CFG["retrieval_ks"], geo_thresholds_km=CFG["geo_thresholds_km"])

    print("\n" + "=" * 60)
    print("§4.4  REPRESENTATION QUALITY")
    print("=" * 60)
    all_results["representation"] = eval_representation_quality(
        backbone_path, eurosat_rgb_dir,
        knn_k=CFG["knn_k"], num_classes=CFG["num_classes"])

    print("\n" + "=" * 60)
    print("§4.5  BAND MISMATCH ROBUSTNESS")
    print("=" * 60)
    all_results["band_mismatch"] = eval_band_mismatch(
        backbone_path, eurosat_rgb_dir, eurosat_ms_dir)

    out_path = os.path.join(CFG["output_dir"], "mocov3_eval_results.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nAll results saved to {out_path}")
    return all_results


# =============================================================================
# ── Entry point ───────────────────────────────────────────────────────────────
# =============================================================================

def is_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False


if __name__ == "__main__":
    world_size = torch.cuda.device_count()

    if world_size > 1 and not is_notebook():
        os.environ["MASTER_ADDR"] = "localhost"
        os.environ["MASTER_PORT"] = "12355"
        mp.spawn(train, args=(world_size,), nprocs=world_size, join=True)
    else:
        if world_size > 1 and is_notebook():
            print(f"Notebook detected — DDP disabled. "
                  f"Running single-GPU on cuda:0. "
                  f"({world_size} GPUs available but only 1 will be used.)")
        train(rank=0, world_size=1)

    BEST_CKPT = os.path.join(CFG["output_dir"],
                              f"mocov3_ep{CFG['epochs']}.pt")
    GPS_CSV   = f"{DATA_DIR}/eurosat/eurosat_gps.csv"
    run_full_evaluation(
        BEST_CKPT,
        CFG["eurosat_rgb_dir"],
        CFG["eurosat_ms_dir"],
        GPS_CSV if os.path.exists(GPS_CSV) else None,
    )


Notebook detected — DDP disabled. Running single-GPU on cuda:0. (2 GPUs available but only 1 will be used.)
Device: cuda:0  |  World: 1  |  Workers: 4  |  compile: True  |  AMP: torch.float16  |  albu: True
torch.compile enabled on student + momentum backbone.


W0417 10:58:33.438000 55 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch [  1/200]  loss=9.5788  lr=3.00e-05  time=182s
Epoch [  2/200]  loss=9.2412  lr=6.00e-05  time=118s
Epoch [  3/200]  loss=9.0395  lr=9.00e-05  time=118s
Epoch [  4/200]  loss=8.8651  lr=1.20e-04  time=118s
Epoch [  5/200]  loss=8.6116  lr=1.50e-04  time=118s
Epoch [  6/200]  loss=8.3286  lr=1.80e-04  time=118s
Epoch [  7/200]  loss=8.0001  lr=2.10e-04  time=119s
Epoch [  8/200]  loss=7.6979  lr=2.40e-04  time=119s
Epoch [  9/200]  loss=7.4350  lr=2.70e-04  time=118s
Epoch [ 10/200]  loss=7.1870  lr=3.00e-04  time=119s
Epoch [ 11/200]  loss=6.9815  lr=3.00e-04  time=119s
Epoch [ 12/200]  loss=6.8068  lr=3.00e-04  time=118s
Epoch [ 13/200]  loss=6.6535  lr=3.00e-04  time=118s
Epoch [ 14/200]  loss=6.5016  lr=3.00e-04  time=119s
Epoch [ 15/200]  loss=6.4147  lr=3.00e-04  time=119s
Epoch [ 16/200]  loss=6.3155  lr=2.99e-04  time=118s
Epoch [ 17/200]  loss=6.1982  lr=2.99e-04  time=119s
Epoch [ 18/200]  loss=6.1241  lr=2.99e-04  time=119s
Epoch [ 19/200]  loss=6.0473  lr=2.99e-04  tim

RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [2]:
# ── Cell: Continue evaluation from §4.5 ──────────────────────────────────────
# Paste this in a new cell. Assumes all functions from the main script are
# already defined in the kernel (the training cell ran successfully).
# Fixes the inference_mode tensor bug in eval_band_mismatch.

def eval_band_mismatch_fixed(backbone_path, eurosat_rgb_dir,
                             eurosat_ms_dir=None, num_classes=10):
    device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    backbone = load_backbone(backbone_path, device)
    split_dir = ensure_split(eurosat_rgb_dir, CFG["val_split"])
    _, val_tf = get_eval_transforms()

    val_ds  = datasets.ImageFolder(
        os.path.join(split_dir, "val"), transform=val_tf)
    val_ldr = make_loader(val_ds, 256, shuffle=False)
    feats, labels = extract_features(backbone, val_ldr, device)
    feats  = F.normalize(feats.float(), dim=-1)
    sim_nd = feats @ feats.T
    sim_nd.fill_diagonal_(-1e9)
    recall_rgb = 100.0 * (
        labels[sim_nd.argmax(dim=1)] == labels).float().mean().item()
    print(f"  Recall@1 (RGB, 3-band) = {recall_rgb:.2f}%")

    recall_ms = None
    if eurosat_ms_dir and os.path.exists(eurosat_ms_dir):
        ms_ds = MultiSpectralDataset(eurosat_ms_dir, n_channels=13)
        if len(ms_ds) == 0:
            print("  EuroSAT-MS: no files found — skipping.")
        else:
            adapter_bb = BandAdapterBackbone(
                backbone, in_channels=13, out_channels=3).to(device)
            ms_ldr_train = make_loader(ms_ds, 128, shuffle=True)

            tmp_head  = nn.Linear(CFG["embed_dim"], num_classes).to(device)
            opt_adapt = torch.optim.AdamW(
                list(adapter_bb.adapter.parameters()) +
                list(tmp_head.parameters()), lr=1e-3)

            for ep in range(5):
                adapter_bb.adapter.train(); tmp_head.train()
                for x, y in ms_ldr_train:
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    # FIX: use torch.no_grad() (not inference_mode) so the
                    # adapter output stays in autograd-tracked territory,
                    # then .clone() to detach backbone grads cleanly.
                    with torch.no_grad():
                        feat = backbone(adapter_bb.adapter(x)).clone()
                    opt_adapt.zero_grad()
                    F.cross_entropy(tmp_head(feat), y).backward()
                    opt_adapt.step()
            del tmp_head

            adapter_bb.eval()
            ms_ldr_eval      = make_loader(ms_ds, 128, shuffle=False)
            ms_feats, ms_lbl = [], []
            with torch.no_grad():
                for x, y in ms_ldr_eval:
                    ms_feats.append(
                        adapter_bb(x.to(device, non_blocking=True))
                        .float().cpu())
                    ms_lbl.append(y)
            ms_feats  = F.normalize(torch.cat(ms_feats), dim=-1)
            ms_labels = torch.cat(ms_lbl)
            sim_ms    = ms_feats @ ms_feats.T
            sim_ms.fill_diagonal_(-1e9)
            recall_ms = 100.0 * (
                ms_labels[sim_ms.argmax(dim=1)] == ms_labels
            ).float().mean().item()
            print(f"  Recall@1 (MS, 13-band) = {recall_ms:.2f}%")
    else:
        print("  EuroSAT-MS directory not found — skipping.")

    delta = round(recall_rgb - recall_ms, 2) if recall_ms is not None else None
    if delta is not None:
        print(f"  Band mismatch penalty Δ = {delta:.2f}%")

    return {
        "recall1_rgb":         round(recall_rgb, 2),
        "recall1_ms":          round(recall_ms, 2) if recall_ms is not None
                               else None,
        "band_mismatch_delta": delta,
    }


# ── Load the previously saved partial results ─────────────────────────────────
import json, os

eval_json_path = os.path.join(CFG["output_dir"], "mocov3_eval_results.json")
if os.path.exists(eval_json_path):
    with open(eval_json_path) as f:
        all_results = json.load(f)
    print("Loaded existing partial results.")
else:
    # Reconstruct from what we already saw in the logs
    all_results = {
        "model": "MoCo-v3-ViT-S",
        "checkpoint": BEST_CKPT,
        "classification": {
            "1pct":   {"top1_acc": 49.70, "macro_f1": 48.60},
            "10pct":  {"top1_acc": 65.37, "macro_f1": 64.24},
            "100pct": {"top1_acc": 71.00, "macro_f1": 69.79},
        },
        "segmentation": {"miou": 46.77, "boundary_f1": 0.00},
        "retrieval": {
            "recall@1": 58.96, "recall@5": 85.52,
            "recall@10": 92.26, "mAP": 26.82,
        },
        "representation": {
            "knn_acc": 67.96, "effective_rank": 3.2,
            "uniformity": -1.0102, "alignment": 0.5819,
        },
    }
    print("No saved JSON found — reconstructed from logs.")

# ── Run §4.5 with the fixed function ─────────────────────────────────────────
BEST_CKPT = os.path.join(CFG["output_dir"], f"mocov3_ep{CFG['epochs']}.pt")
GPS_CSV   = f"{DATA_DIR}/eurosat/eurosat_gps.csv"

print("\n" + "="*60)
print("§4.5  BAND MISMATCH ROBUSTNESS")
print("="*60)
all_results["band_mismatch"] = eval_band_mismatch_fixed(
    BEST_CKPT,
    CFG["eurosat_rgb_dir"],
    CFG["eurosat_ms_dir"],
)

# ── Save final merged results ─────────────────────────────────────────────────
out_path = os.path.join(CFG["output_dir"], "mocov3_eval_results.json")
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nFinal results saved to {out_path}")
print(json.dumps(all_results, indent=2))

No saved JSON found — reconstructed from logs.

§4.5  BAND MISMATCH ROBUSTNESS
  Recall@1 (RGB, 3-band) = 57.93%
  Recall@1 (MS, 13-band) = 34.57%
  Band mismatch penalty Δ = 23.36%

Final results saved to /kaggle/working/mocov3/mocov3_eval_results.json
{
  "model": "MoCo-v3-ViT-S",
  "checkpoint": "/kaggle/working/mocov3/mocov3_ep200.pt",
  "classification": {
    "1pct": {
      "top1_acc": 49.7,
      "macro_f1": 48.6
    },
    "10pct": {
      "top1_acc": 65.37,
      "macro_f1": 64.24
    },
    "100pct": {
      "top1_acc": 71.0,
      "macro_f1": 69.79
    }
  },
  "segmentation": {
    "miou": 46.77,
    "boundary_f1": 0.0
  },
  "retrieval": {
    "recall@1": 58.96,
    "recall@5": 85.52,
    "recall@10": 92.26,
    "mAP": 26.82
  },
  "representation": {
    "knn_acc": 67.96,
    "effective_rank": 3.2,
    "uniformity": -1.0102,
    "alignment": 0.5819
  },
  "band_mismatch": {
    "recall1_rgb": 57.93,
    "recall1_ms": 34.57,
    "band_mismatch_delta": 23.36
  }
}
